# Exploratory Data Analysis: U.S. COVID-19

**Project:** Modeling and Forecasting COVID-19 Dynamics in the United States Using Logistic Growth Models

This notebook explores the cleaned, weekly U.S. COVID-19 dataset produced by `data_cleaning.ipynb`. It summarizes the data, checks the reporting window, and visualizes cumulative cases, weekly new cases, and weekly new deaths.

**Important:** These plots describe the full U.S. epidemic history. We will not fit one logistic curve to the entire cumulative series because it contains multiple epidemic waves. Later analysis will identify individual waves and fit a separate curve to each one.

**How to run:** Run all cells in order. The data is loaded from this repository's GitHub raw URL, so no local file path or manual upload is needed.

## 1. Imports and data loading

The source table contains one row per week-ending Sunday. The `cases_reported` flag identifies the weeks for which U.S. case reporting was still available. Death observations continue beyond that date.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "troyc03/cpsc-375-project/main/data/processed/us_covid_clean.csv"
)

df = pd.read_csv(DATA_URL, parse_dates=["week_ending"])
df = df.sort_values("week_ending").reset_index(drop=True)

required_columns = {
    "week_ending", "total_cases", "total_deaths",
    "new_cases", "new_deaths", "week", "cases_reported"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

df.head()

## 2. Basic data checks

We verify the date range, number of weekly observations, missing values, duplicate dates, and whether the case-reporting flag is present.

In [ ]:
print(f"Rows: {len(df)}")
print(f"Date range: {df['week_ending'].min().date()} to {df['week_ending'].max().date()}")
print(f"Duplicate week-ending dates: {df['week_ending'].duplicated().sum()}")
print("\nMissing values:")
print(df[sorted(required_columns)].isna().sum())

print("\nCase reporting status:")
print(df["cases_reported"].value_counts(dropna=False))

case_window = df.loc[df["cases_reported"].astype(bool)].copy()
print(
    "\nLast week with reported cases:",
    case_window["week_ending"].max().date()
)

## 3. Summary statistics

Weekly new cases are restricted to the period in which case reporting was available. Weekly deaths are retained across the full available death-reporting period.

In [ ]:
case_window[["total_cases", "new_cases"]].describe().round(2)

In [ ]:
df[["total_deaths", "new_deaths"]].describe().round(2)

## 4. Cumulative cases and weekly new cases

The cumulative series is useful for understanding the overall scale of the epidemic. Weekly new cases show the timing and relative size of waves more clearly.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

axes[0].plot(
    case_window["week_ending"],
    case_window["total_cases"] / 1_000_000,
    linewidth=2
)
axes[0].set_title("Cumulative Reported COVID-19 Cases in the United States")
axes[0].set_ylabel("Cumulative cases (millions)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    case_window["week_ending"],
    case_window["new_cases"] / 1_000_000,
    linewidth=1.5
)
axes[1].set_title("Weekly New Reported COVID-19 Cases in the United States")
axes[1].set_xlabel("Week ending")
axes[1].set_ylabel("New cases (millions)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Weekly new deaths

Deaths continue to be available after U.S. case reporting stops, so this plot uses the full cleaned dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    df["week_ending"],
    df["new_deaths"] / 1_000,
    linewidth=1.5
)
ax.set_title("Weekly New Reported COVID-19 Deaths in the United States")
ax.set_xlabel("Week ending")
ax.set_ylabel("New deaths (thousands)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Largest reported case weeks

This table identifies the weeks with the highest reported case counts. These peaks are useful reference points for the next notebook, where we will identify and label individual epidemic waves.

In [ ]:
top_case_weeks = (
    case_window.nlargest(10, "new_cases")[
        ["week_ending", "new_cases", "total_cases"]
    ]
    .sort_values("new_cases", ascending=False)
    .reset_index(drop=True)
)
top_case_weeks

## 7. Interpretation and next steps

- The cumulative case series increases over time, but its slope changes as new epidemic waves occur.
- Weekly new cases make separate waves easier to identify than cumulative cases alone.
- The case-reporting flag must be respected: the frozen cumulative case total after the last reported week is not evidence that new cases were zero.
- A single logistic curve is not appropriate for the entire multi-wave cumulative series.

**Next:** use weekly new cases to define epidemic-wave start, peak, and end dates. Then construct cumulative cases within each wave and fit a separate continuous logistic model to each wave.